# 03bis — Terrain : détection des portions en forêt avec OpenStreetMap

Ce notebook est **complémentaire du notebook 03**.

```text
Notebook 03
→ géométrie/topographie issue du GPX

Notebook 03bis
→ contexte terrain issu d'une source géographique externe
```

Pour cette première version terrain, une seule caractéristique est ajoutée :

```text
foret
```

Sources OSM utilisées :

```text
natural=wood
landuse=forest
landcover=trees
```

L'OpenStreetMap Wiki indique que `natural=wood` et `landuse=forest`
sont très souvent traités par les consommateurs comme des zones boisées.
`landcover=trees` est ajouté comme preuve explicite de couverture arborée.

Limite importante :
l'absence de forêt dans OSM ne prouve pas l'absence réelle de forêt.
Ce notebook produit donc une annotation basée sur la couverture OSM disponible.

In [ ]:
import json
import math
from pathlib import Path

import numpy as np
import pandas as pd
import requests
import matplotlib.pyplot as plt

from pyproj import Transformer
from shapely.geometry import (
    LineString,
    Polygon,
    MultiPolygon,
)
from shapely.ops import (
    transform,
    unary_union,
    polygonize,
)

## 1. Charger la route du notebook 02

In [ ]:
route_path = Path("data/output/route_resampled.csv")

if not route_path.exists():
    raise FileNotFoundError(
        "Exécute d'abord le notebook 02."
    )

route = pd.read_csv(route_path)

print("Route :", route_path)
print("Points :", len(route))
print("Longueur :", round(route["distance_m"].iloc[-1], 1), "m")

## 2. Configuration Overpass / forêt

Le service Overpass est un service public partagé.
Cette V1 fait une seule requête autour d'un itinéraire.

La marge de bbox sert à éviter de requêter uniquement l'axe exact de la route.

In [ ]:
OVERPASS_URL = "https://overpass-api.de/api/interpreter"

# Environ 2 km de marge en latitude.
# Valeur simple pour la V1 ; ce n'est pas une distance métrique exacte.
BBOX_MARGIN_DEG = 0.02

# Rayon du corridor autour de chaque petit segment de route.
CORRIDOR_RADIUS_M = 10.0

# Un segment est classé "foret" si au moins 50 % de son corridor
# intersecte une zone boisée OSM.
FOREST_COVERAGE_THRESHOLD = 0.50

## 3. Construire la bbox et la requête Overpass

In [ ]:
south = float(route["lat"].min() - BBOX_MARGIN_DEG)
north = float(route["lat"].max() + BBOX_MARGIN_DEG)
west = float(route["lon"].min() - BBOX_MARGIN_DEG)
east = float(route["lon"].max() + BBOX_MARGIN_DEG)

bbox = f"{south},{west},{north},{east}"

overpass_query = (
    f'[out:json][timeout:90];\n'
    f'(\n'
    f'  way["natural"="wood"]({bbox});\n'
    f'  relation["natural"="wood"]({bbox});\n'
    f'  way["landuse"="forest"]({bbox});\n'
    f'  relation["landuse"="forest"]({bbox});\n'
    f'  way["landcover"="trees"]({bbox});\n'
    f'  relation["landcover"="trees"]({bbox});\n'
    f');\n'
    f'out tags geom;'
)

print("BBox :", bbox)
print(overpass_query)


## 4. Télécharger les objets boisés OSM

Si Overpass est temporairement surchargé, relancer cette cellule plus tard
ou changer d'instance Overpass.

In [ ]:
headers = {
    "User-Agent": "PIR-Route-Agent/0.1 educational-research"
}

response = requests.post(
    OVERPASS_URL,
    data={"data": overpass_query},
    headers=headers,
    timeout=120,
)

if response.status_code != 200:
    raise RuntimeError(
        f"Overpass HTTP {response.status_code}: "
        f"{response.text[:500]}"
    )

osm_data = response.json()

print(
    "Éléments OSM reçus :",
    len(osm_data.get("elements", []))
)

## 5. Convertir les objets OSM en polygones

Les `way` fermés sont convertis directement en polygones.

Pour les relations multipolygones, les morceaux `outer` sont assemblés
avec `polygonize`; les morceaux `inner` sont retirés si possible.

In [ ]:
def coords_from_geometry(geometry):
    if not geometry:
        return []

    return [
        (float(p["lon"]), float(p["lat"]))
        for p in geometry
        if "lon" in p and "lat" in p
    ]


def polygon_from_way_geometry(geometry):
    coords = coords_from_geometry(geometry)

    if len(coords) < 4:
        return None

    if coords[0] != coords[-1]:
        return None

    polygon = Polygon(coords)

    if polygon.is_empty:
        return None

    if not polygon.is_valid:
        polygon = polygon.buffer(0)

    return polygon if not polygon.is_empty else None


def polygon_from_relation(element):
    outer_lines = []
    inner_lines = []

    for member in element.get("members", []):
        geometry = member.get("geometry")
        coords = coords_from_geometry(geometry)

        if len(coords) < 2:
            continue

        line = LineString(coords)
        role = member.get("role", "")

        if role == "inner":
            inner_lines.append(line)
        else:
            outer_lines.append(line)

    if not outer_lines:
        return None

    outer_polygons = list(
        polygonize(unary_union(outer_lines))
    )

    if not outer_polygons:
        return None

    result = unary_union(outer_polygons)

    if inner_lines:
        inner_polygons = list(
            polygonize(unary_union(inner_lines))
        )

        if inner_polygons:
            result = result.difference(
                unary_union(inner_polygons)
            )

    if not result.is_valid:
        result = result.buffer(0)

    return result if not result.is_empty else None


forest_geometries = []

for element in osm_data.get("elements", []):
    geometry = None

    if element.get("type") == "way":
        geometry = polygon_from_way_geometry(
            element.get("geometry")
        )

    elif element.get("type") == "relation":
        geometry = polygon_from_relation(element)

    if geometry is not None and not geometry.is_empty:
        forest_geometries.append(geometry)

if not forest_geometries:
    print(
        "Aucun polygone boisé exploitable trouvé dans OSM "
        "autour de cette route."
    )
    forest_wgs84 = None
else:
    forest_wgs84 = unary_union(forest_geometries)

    if not forest_wgs84.is_valid:
        forest_wgs84 = forest_wgs84.buffer(0)

    print(
        "Géométries forestières exploitables :",
        len(forest_geometries)
    )

## 6. Projection métrique locale

Pour calculer un corridor de 10 m, il faut travailler dans un CRS métrique.

La V1 choisit automatiquement une zone UTM à partir du centre de l'itinéraire.

In [ ]:
def utm_epsg_for_lonlat(lon, lat):
    zone = int((lon + 180) / 6) + 1

    if lat >= 0:
        return 32600 + zone

    return 32700 + zone


center_lon = float(route["lon"].mean())
center_lat = float(route["lat"].mean())

utm_epsg = utm_epsg_for_lonlat(
    center_lon,
    center_lat,
)

transformer = Transformer.from_crs(
    "EPSG:4326",
    f"EPSG:{utm_epsg}",
    always_xy=True,
)

project_geom = lambda geom: transform(
    transformer.transform,
    geom,
)

route_line_wgs84 = LineString(
    list(zip(route["lon"], route["lat"]))
)

route_line_m = project_geom(route_line_wgs84)

forest_m = (
    project_geom(forest_wgs84)
    if forest_wgs84 is not None
    else None
)

print("CRS métrique :", f"EPSG:{utm_epsg}")

## 7. Couverture forestière sur chaque intervalle de la route

La route est déjà échantillonnée approximativement tous les 10 m.

Pour chaque intervalle :

```text
point_i -------- point_i+1
         ↓
    corridor ±10 m
         ↓
intersection avec les polygones OSM
         ↓
forest_coverage ∈ [0,1]
```

In [ ]:
projected_points = [
    transformer.transform(lon, lat)
    for lon, lat in zip(
        route["lon"].to_numpy(),
        route["lat"].to_numpy(),
    )
]

segment_rows = []

for i in range(len(route) - 1):
    start_m = float(route.loc[i, "distance_m"])
    end_m = float(route.loc[i + 1, "distance_m"])

    segment = LineString([
        projected_points[i],
        projected_points[i + 1],
    ])

    corridor = segment.buffer(
        CORRIDOR_RADIUS_M,
        cap_style=2,
    )

    if (
        forest_m is None
        or forest_m.is_empty
        or corridor.area <= 0
    ):
        coverage = 0.0
    else:
        coverage = (
            corridor.intersection(forest_m).area
            / corridor.area
        )

    segment_rows.append({
        "start_m": start_m,
        "end_m": end_m,
        "forest_coverage": float(
            max(0.0, min(1.0, coverage))
        ),
        "is_forest": bool(
            coverage >= FOREST_COVERAGE_THRESHOLD
        ),
    })

forest_segments = pd.DataFrame(segment_rows)

display(forest_segments.head(20))

## 8. Visualiser la couverture forestière

In [ ]:
fig, ax = plt.subplots()

x = (
    forest_segments["start_m"]
    + forest_segments["end_m"]
) / 2

ax.plot(
    x,
    forest_segments["forest_coverage"],
)

ax.axhline(
    FOREST_COVERAGE_THRESHOLD,
    linestyle="--",
)

ax.set_xlabel("Distance le long de l'itinéraire (m)")
ax.set_ylabel("Couverture forestière OSM")
ax.set_ylim(-0.05, 1.05)
ax.set_title("Couverture forestière le long de l'itinéraire")

plt.show()

## 9. Regrouper les segments forestiers contigus

Plusieurs intervalles de 10 m successifs deviennent une annotation `foret`
plus longue.

Exemple :

```text
100–110 forêt
110–120 forêt
120–130 forêt

→

100–130 forêt
```

In [ ]:
forest_annotations = []
current = None

for _, row in forest_segments.iterrows():
    if not bool(row["is_forest"]):
        if current is not None:
            forest_annotations.append(current)
            current = None
        continue

    evidence = {
        "start_m": float(row["start_m"]),
        "end_m": float(row["end_m"]),
        "coverage": float(row["forest_coverage"]),
    }

    if current is None:
        current = {
            "feature": "foret",
            "family": "terrain",
            "start_m": float(row["start_m"]),
            "end_m": float(row["end_m"]),
            "resolution_m": float(
                row["end_m"] - row["start_m"]
            ),
            "confidence_values": [
                float(row["forest_coverage"])
            ],
            "evidence": [evidence],
        }

    elif abs(
        float(row["start_m"]) - current["end_m"]
    ) < 1e-6:
        current["end_m"] = float(row["end_m"])
        current["confidence_values"].append(
            float(row["forest_coverage"])
        )
        current["evidence"].append(evidence)

    else:
        forest_annotations.append(current)
        current = {
            "feature": "foret",
            "family": "terrain",
            "start_m": float(row["start_m"]),
            "end_m": float(row["end_m"]),
            "resolution_m": float(
                row["end_m"] - row["start_m"]
            ),
            "confidence_values": [
                float(row["forest_coverage"])
            ],
            "evidence": [evidence],
        }

if current is not None:
    forest_annotations.append(current)

final_forest_annotations = []

for item in forest_annotations:
    values = item.pop("confidence_values")

    final_forest_annotations.append({
        "feature": "foret",
        "family": "terrain",
        "start_m": item["start_m"],
        "end_m": item["end_m"],

        # Le détecteur terrain travaille à la résolution spatiale
        # de base de la route.
        "resolution_m": 10.0,

        "confidence": float(np.mean(values)),
        "metrics": {
            "source": "OpenStreetMap",
            "mean_forest_coverage": float(np.mean(values)),
            "min_forest_coverage": float(np.min(values)),
            "max_forest_coverage": float(np.max(values)),
            "corridor_radius_m": CORRIDOR_RADIUS_M,
            "coverage_threshold": FOREST_COVERAGE_THRESHOLD,
            "n_base_segments": len(item["evidence"]),
        },
    })

print(
    "Portions forestières détectées :",
    len(final_forest_annotations)
)

display(pd.DataFrame(final_forest_annotations))

## 10. Export pour le notebook 04

In [ ]:
output_dir = Path("data/output")
output_dir.mkdir(parents=True, exist_ok=True)

forest_json_path = (
    output_dir / "terrain_forest_annotations.json"
)
forest_csv_path = (
    output_dir / "terrain_forest_annotations.csv"
)

with forest_json_path.open("w", encoding="utf-8") as f:
    json.dump(
        final_forest_annotations,
        f,
        ensure_ascii=False,
        indent=2,
    )

pd.DataFrame([
    {
        **{k: v for k, v in a.items() if k != "metrics"},
        "metrics_json": json.dumps(
            a["metrics"],
            ensure_ascii=False,
        ),
    }
    for a in final_forest_annotations
]).to_csv(
    forest_csv_path,
    index=False,
)

print("Export terrain terminé :")
print("-", forest_json_path)
print("-", forest_csv_path)

## Limites de cette V1 forêt

- OSM peut être incomplet localement.
- Les limites forestières OSM peuvent être imprécises.
- Une bbox élargie est utilisée pour cette V1.
- Une route située juste à côté d'une forêt peut être classée partiellement
  en forêt à cause du corridor de 10 m.
- `foret` signifie ici **couverture boisée OSM**, pas ombre garantie.

Ces limites doivent être conservées dans l'interprétation scientifique.